# Lesson 15: Long-Term Memory — Persist Calculations Across Sessions

## Objective
Store past arithmetic calculations in a **persistent store** (SQLite) so the agent can recall results from previous sessions — true long-term memory.

## Limitation of Lesson 14
- ❌ Memory was in-process only — lost when Python restarts
- ❌ Can't recall "last Monday I computed 45 × 3 = 135"
- ❌ No cross-session continuity

## What is NEW in Lesson 15?
- ✅ **Long-term memory store** using SQLite (via `sqlite3`)
- ✅ **Memory write node**: saves each computation to the database
- ✅ **Memory read node**: retrieves relevant past calculations at session start
- ✅ **Semantic recall pattern**: find similar past calculations
- ✅ Cross-session persistence

## Theory: Long-Term Memory Architecture

```
Session 1:           Session 2:
  Ask "5 + 3?"         Ask "what was my last addition?"
  Compute: 8           Memory read → "5 + 3 = 8 (yesterday)"
  Store to DB ──────→  LLM answers using retrieved memory
```

LangGraph has a built-in `InMemoryStore` / `PostgresStore` for production.  
In this lesson we build the pattern manually with SQLite to understand the mechanics.

### Memory Types
| Type | Storage | Scope |
|------|---------|-------|
| Short-term | messages list | Current session |
| Checkpoint | Checkpointer DB | Per thread |
| Long-term | External DB | All sessions |


In [ ]:
# ── Cell 1: Imports + Setup ───────────────────────────────────────────────────
from dotenv import load_dotenv
import warnings
warnings.filterwarnings("ignore")

import os, sqlite3, json as json_lib, datetime
import vertexai
from langchain_google_vertexai import ChatVertexAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

from langgraph.graph import StateGraph, MessagesState, START, END
from typing import TypedDict, Optional, Annotated
from langgraph.graph.message import add_messages
from IPython.display import Image, display

load_dotenv()
vertexai.init(project=os.getenv("PROJECT_ID"), location=os.getenv("LOCATION"))
llm = ChatVertexAI(model="gemini-2.5-flash", temperature=0)

# Long-term memory database path
MEMORY_DB = "/tmp/arithmetic_memory.db"
print(f"✓ Setup complete | Memory DB: {MEMORY_DB}")


In [ ]:
# ── Cell 2: Long-Term Memory Store (SQLite) ──────────────────────────────────

class ArithmeticMemoryStore:
    """
    Simple SQLite-backed long-term memory for arithmetic calculations.
    Stores: question, a, b, operation, result, timestamp.
    In production: use LangGraph's InMemoryStore, PostgresStore, or a vector DB.
    """
    
    def __init__(self, db_path: str):
        self.db_path = db_path
        self._init_db()
    
    def _init_db(self):
        with sqlite3.connect(self.db_path) as conn:
            conn.execute("""
                CREATE TABLE IF NOT EXISTS calculations (
                    id        INTEGER PRIMARY KEY AUTOINCREMENT,
                    question  TEXT NOT NULL,
                    operation TEXT,
                    a         REAL,
                    b         REAL,
                    result    REAL,
                    timestamp TEXT
                )
            """)
            conn.commit()
    
    def save(self, question: str, operation: str, a: float, b: float, result: float):
        ts = datetime.datetime.now().isoformat()
        with sqlite3.connect(self.db_path) as conn:
            conn.execute(
                "INSERT INTO calculations (question, operation, a, b, result, timestamp) VALUES (?,?,?,?,?,?)",
                (question, operation, a, b, result, ts)
            )
            conn.commit()
        print(f"  [memory:save] Stored: {question} → {result}")
    
    def recall_recent(self, n: int = 5) -> list[dict]:
        with sqlite3.connect(self.db_path) as conn:
            rows = conn.execute(
                "SELECT question, operation, a, b, result, timestamp FROM calculations ORDER BY id DESC LIMIT ?", (n,)
            ).fetchall()
        return [{"question":r[0],"operation":r[1],"a":r[2],"b":r[3],"result":r[4],"ts":r[5]} for r in rows]
    
    def recall_by_operation(self, operation: str, n: int = 3) -> list[dict]:
        with sqlite3.connect(self.db_path) as conn:
            rows = conn.execute(
                "SELECT question, operation, a, b, result, timestamp FROM calculations WHERE operation=? ORDER BY id DESC LIMIT ?",
                (operation, n)
            ).fetchall()
        return [{"question":r[0],"operation":r[1],"a":r[2],"b":r[3],"result":r[4],"ts":r[5]} for r in rows]
    
    def clear(self):
        with sqlite3.connect(self.db_path) as conn:
            conn.execute("DELETE FROM calculations")
            conn.commit()

# Initialize the store
memory_store = ArithmeticMemoryStore(MEMORY_DB)
print(f"✓ ArithmeticMemoryStore initialized at {MEMORY_DB}")


In [ ]:
# ── Cell 3: State Schema ────────────────────────────────────────────────────
class State(MessagesState):
    user_request: str
    a: Optional[float]
    b: Optional[float]
    operation: Optional[str]
    result: Optional[float]
    recalled_memories: list    # Memories retrieved at session start

print("✓ State defined")


In [ ]:
# ── Cell 4: Memory Read Node ─────────────────────────────────────────────────
# Runs at the START of each turn
# Retrieves relevant past calculations and puts them in state

def memory_read_node(state: State) -> dict:
    recent = memory_store.recall_recent(5)
    print(f"  [memory:read] Retrieved {len(recent)} recent calculations")
    for r in recent:
        print(f"    • {r['question']} = {r['result']} ({r['ts'][:10]})")
    return {"recalled_memories": recent}

print("✓ Memory read node defined")


In [ ]:
# ── Cell 5: Agent Node with Long-Term Context ────────────────────────────────
SYSTEM_PROMPT = """You are an arithmetic assistant with long-term memory.
You can recall past calculations provided in your context.
Always end with RESULT: <number>
If user asks about past calculations, use the provided memory.
"""

def arithmetic_agent(state: State) -> dict:
    memories = state.get("recalled_memories", [])
    
    # Build memory context string
    memory_context = ""
    if memories:
        memory_context = "Past calculations you remember:\n"
        for m in memories:
            memory_context += f"  - {m['question']} = {m['result']} (on {m['ts'][:10]})\n"
    
    context = [SystemMessage(content=SYSTEM_PROMPT)]
    if memory_context:
        context.append(SystemMessage(content=memory_context))
    context.extend(state["messages"])
    
    print(f"  [agent] Processing with {len(memories)} memories in context")
    response = llm.invoke(context)
    ai_content = response.content.strip()
    print(f"  [agent] {ai_content[:100]}")
    
    # Extract RESULT
    result = state.get("result")
    operation = "add"
    for line in ai_content.split("\n"):
        if "RESULT:" in line:
            try: result = float(line.split("RESULT:")[1].strip())
            except: pass
    
    return {"messages": [AIMessage(content=ai_content)], "result": result}

print("✓ Agent with long-term context defined")


In [ ]:
# ── Cell 6: Memory Write Node ─────────────────────────────────────────────────
# Runs AFTER the agent
# Saves the computation to long-term storage

def memory_write_node(state: State) -> dict:
    result = state.get("result")
    request = state["messages"][-2].content if len(state["messages"]) >= 2 else ""
    
    if result is not None and request:
        # Save to persistent store
        memory_store.save(
            question=request,
            operation=state.get("operation", "unknown"),
            a=state.get("a", 0),
            b=state.get("b", 0),
            result=result
        )
    return {}   # No state changes

print("✓ Memory write node defined")


In [ ]:
# ── Cell 7: Build Graph ──────────────────────────────────────────────────────
builder = StateGraph(State)

builder.add_node("memory_read",  memory_read_node)
builder.add_node("agent",        arithmetic_agent)
builder.add_node("memory_write", memory_write_node)

# Flow: read memory → agent → write memory
builder.add_edge(START,          "memory_read")
builder.add_edge("memory_read",  "agent")
builder.add_edge("agent",        "memory_write")
builder.add_edge("memory_write", END)

graph = builder.compile()
print("✓ Long-term memory graph compiled")


In [ ]:
# ── Cell 8: Visualize ───────────────────────────────────────────────────────
display(Image(graph.get_graph().draw_mermaid_png()))


In [ ]:
# ── Cell 9: Session 1 — Do some calculations ─────────────────────────────────
memory_store.clear()  # Fresh start for demo

def run(question: str):
    state = {
        "user_request": question,
        "messages": [HumanMessage(content=question)],
        "a": None, "b": None, "operation": None,
        "result": None, "recalled_memories": []
    }
    print(f"\nQ: {question}")
    out = graph.invoke(state)
    print(f"A: {out['messages'][-1].content[:80]}")
    return out

print("=== Session 1: Building memory ===")
run("what is 25 plus 17?")
run("multiply 8 by 6")
run("divide 144 by 12")


In [ ]:
# ── Cell 10: Session 2 — Recall past calculations ────────────────────────────
print("\n=== Session 2: Recalling past work ===")
print("(Same memory_store, new session — simulates restarting the app)")
print()

# Check what's in memory
print("Long-term memory contents:")
for m in memory_store.recall_recent(10):
    print(f"  • {m['question']} → {m['result']} ({m['ts'][:19]})")

print()
run("what arithmetic have I done before?")
run("what was the result of my last multiplication?")


## Production Long-Term Memory with LangGraph Store

LangGraph provides built-in stores:

```python
from langgraph.store.memory import InMemoryStore
from langgraph.store.postgres import AsyncPostgresStore  # production

store = InMemoryStore()

# In a node — access the store via RunnableConfig
def my_node(state, *, store):  # store injected by LangGraph
    # Read
    items = store.search(("user", "calculations"), query="addition")
    # Write
    store.put(("user", "calculations"), "calc_001", {"result": 42})
```

This is the **official LangGraph way** — your store is accessible via dependency injection in nodes.

## Summary

| Memory Type | Lesson | Persistence | Scope |
|-------------|--------|-------------|-------|
| Short-term messages | 13 | In-process only | Session |
| Trimmed + summarized | 14 | In-process only | Session |
| Long-term SQLite | 15 | Cross-session | All threads |

## Limitations → What Lesson 16 Solves
- ❌ `invoke()` returns only the final state — we see nothing until it's done
- ❌ For long-running agents, users see no progress indicator
- ❌ **Lesson 16**: `stream()` to receive real-time updates as each node runs
